In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/medgemma-4b-it",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,              # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    processor,
    chat_template = "gemma-3",
)

In [ ]:
from datasets import load_dataset, concatenate_datasets, Image

# --- 1. Load each dataset ---
dataset_1 = load_dataset("Ahmed-Selem/Shifaa_Arabic_Medical_Consultations", split="train")
dataset_2 = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'en', split="train")
dataset_3 = load_dataset("RecurvAI/Recurv-Medical-Dataset", split="train")
# ROCOv2 contains 'image' and 'caption'
dataset_4 = load_dataset("unsloth/Radiology_mini", split="train")
dataset_5 = load_dataset("Lunzima/Radiology", split="train")
# --- 2. Define Extraction Functions ---

# For Text Datasets: Add a None image column
def format_text_ds1(example):
    return {"input": example["Question"], "output": example["Answer"], "image": None}

def format_text_ds2(example):
    return {"input": example["Question"], "output": example["Response"], "image": None}

def format_text_ds3(example):
    return {"input": example["input"], "output": example["output"], "image": None}

# For Image Datasets with 'image' and 'caption' columns (like dataset_4)
def format_image_ds_caption(example):
    # For datasets with 'caption' column
    try:
        return {
            "input": "Describe this medical image.",
            "output": example["caption"],
            "image": example["image"] # This keeps the PIL Image object
        }
    except KeyError as e:
        print(f"KeyError in format_image_ds_caption: {e}. Available keys: {example.keys()}")
        raise

# For Image Datasets with 'image', 'question', and 'answer' columns (like dataset_5)
def format_image_ds_qa(example):
    # For datasets with 'question' and 'answer' columns
    try:
        return {
            "input": example["question"],
            "output": example["answer"],
            "image": example["image"] # This keeps the PIL Image object
        }
    except KeyError as e:
        print(f"KeyError in format_image_ds_qa: {e}. Available keys: {example.keys()}")
        raise

# --- 3. Apply Mapping ---
# Important: We cast the image column to the Image feature type so they match
dataset_1_clean = dataset_1.map(format_text_ds1, remove_columns=dataset_1.column_names, load_from_cache_file=False)
dataset_2_clean = dataset_2.map(format_text_ds2, remove_columns=dataset_2.column_names, load_from_cache_file=False)
dataset_3_clean = dataset_3.map(format_text_ds3, remove_columns=dataset_3.column_names, load_from_cache_file=False)
dataset_4_clean = dataset_4.map(format_image_ds_caption, remove_columns=dataset_4.column_names, load_from_cache_file=False)
dataset_5_clean = dataset_5.map(format_image_ds_qa, remove_columns=dataset_5.column_names, load_from_cache_file=False)
# --- 4. Merge and Shuffle ---
# We ensure all datasets follow the same feature schema
merged_dataset = concatenate_datasets([
    dataset_1_clean,
    dataset_2_clean,
    dataset_3_clean,
    dataset_4_clean,
    dataset_5_clean
])

med_dataset = merged_dataset.shuffle(seed=42)

# --- 5. Verify ---
print(f"Total samples: {len(med_dataset)}")
print(f"First sample keys: {med_dataset[0].keys()}")
# Check if the first image exists (it might be None if it's a text sample)
print(f"Sample 0 Image: {med_dataset[0]['image']}")

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(med_dataset)

In [ ]:
def formatting_prompts_func(examples):
    # 1. Get the lists from the batch
    # Unsloth's standardize_data_formats usually renames 'input' to 'instruction'
    # We use .get() to be safe if the column is named 'input' or 'instruction'
    inputs  = examples.get('instruction', examples.get('input'))
    outputs = examples['output']

    # CRITICAL: Retrieve the images. If a row has no image, we use None.
    images  = examples.get('image', [None] * len(inputs))

    texts = []
    for i in range(len(inputs)):
        # 2. Construct the User Content (Multimodal Format)
        if images[i] is not None:
            # This tells the model: "Here is an image, and here is the text."
            user_content = [{"type": "image"}, {"type": "text", "text": inputs[i]}]
        else:
            # Pure text example (no image)
            user_content = inputs[i]

        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": outputs[i]} # Use 'assistant', not 'model'
        ]

        # 3. Apply the Chat Template
        formatted_text = tokenizer.apply_chat_template(
            conversation,
            tokenize = False,
            add_generation_prompt = False
        )
        # Remove the BOS token if it's already added by the trainer
        texts.append(formatted_text.lstrip('<|begin_of_text|>').lstrip('<bos>'))

    # 4. CRITICAL: Return BOTH text and image columns
    return { "text" : texts, "image": images }

# Apply the mapping
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "What's the active indgredent in paracetamol?",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)
outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged("MedGemma-4B-it-finetuned", tokenizer)

In [ ]:
if True: # Change to True to upload finetune
    model.push_to_hub_merged(
        "2kfi/MedGemma-4B-it-finetuned_V2.0", tokenizer,
        token = "TOKEN"
    )